# 🔴 Solution: Tensor Parallel MLP

Megatron-style column-parallel + row-parallel MLP

In [ ]:
import torch
import torch.nn as nn


In [ ]:
# ✅ SOLUTION

class TensorParallelMLP(nn.Module):
    """
    Megatron 风格的张量并行 MLP。

    核心思想：
    - 第一层 (上投影 W1) 是列并行：按列分片，每个设备计算 d_ff/world_size 个隐藏维度
    - 第二层 (下投影 W2) 是行并行：按行分片，每个设备接收部分隐藏维度，产生完整输出
    - 最后 all-reduce (求和) 得到最终结果

    参数:
        d_model: 输入/输出维度
        d_ff: 隐藏维度 (必须被 world_size 整除)
        world_size: 虚拟分片数
    """
    def __init__(self, d_model, d_ff, world_size):
        super().__init__()
        self.world_size = world_size
        shard_size = d_ff // world_size

        # ---- Step 1: W1 按列分片 (Column Parallel) ----
        # 完整 W1: (d_model, d_ff)
        # 每个分片 w1_shards[i]: (d_model, d_ff/world_size)
        # 列并行：每个设备只计算输出的一部分列
        self.w1_shards = nn.ParameterList([
            nn.Parameter(torch.randn(d_model, shard_size) * (2 / d_model) ** 0.5)
            for _ in range(world_size)
        ])

        # ---- Step 2: W2 按行分片 (Row Parallel) ----
        # 完整 W2: (d_ff, d_model)
        # 每个分片 w2_shards[i]: (d_ff/world_size, d_model)
        # 行并行：每个设备接收输入的一部分行，产生完整输出
        self.w2_shards = nn.ParameterList([
            nn.Parameter(torch.randn(shard_size, d_model) * (2 / d_ff) ** 0.5)
            for _ in range(world_size)
        ])

    def forward(self, x):
        """
        x: (B, d_model) → (B, d_model)

        流程:
        1. 每个分片 i: h_i = GELU(x @ w1_i)   [列并行]
        2. 每个分片 i: partial_i = h_i @ w2_i   [行并行]
        3. all-reduce: output = sum(partial_i)   [所有分片求和]
        """
        output = None

        for w1, w2 in zip(self.w1_shards, self.w2_shards):
            # ---- Step 3: 列并行前向 ----
            # x: (B, d_model) @ w1: (d_model, shard) = (B, shard)
            # GELU 激活：max(0, x) + 负值部分的平滑近似
            h = torch.nn.functional.gelu(x @ w1)

            # ---- Step 4: 行并行前向 ----
            # h: (B, shard) @ w2: (shard, d_model) = (B, d_model)
            # 每个分片产生完整的 d_model 维输出
            partial = h @ w2

            # ---- Step 5: All-reduce (求和) ----
            # 累加所有分片的输出
            # 在真实分布式中，这一步是 all-reduce 通信
            output = partial if output is None else output + partial

        return output

In [ ]:
# Verify: compare TensorParallelMLP against a standard single-GPU MLP
torch.manual_seed(42)
d_model, d_ff, world_size = 64, 256, 4
x = torch.randn(2, d_model)

tp_mlp = TensorParallelMLP(d_model, d_ff, world_size)
out = tp_mlp(x)
print("Output shape:", out.shape)  # expect (2, 64)

# Build equivalent full MLP from the sharded weights
w1_full = torch.cat([p.data for p in tp_mlp.w1_shards], dim=1)  # (d_model, d_ff)
w2_full = torch.cat([p.data for p in tp_mlp.w2_shards], dim=0)  # (d_ff, d_model)
ref = torch.nn.functional.gelu(x @ w1_full) @ w2_full

print("Max diff vs reference:", (out - ref).abs().max().item())  # expect ~0

In [ ]:
from torch_judge import check
check("tensor_parallel")

## 📝 核心思路总结

1. **列并行 + 行并行**：W1 按列切（每个设备算部分隐藏维度），W2 按行切（每个设备接收部分输入），两者配合只需一次 all-reduce。
2. **数学等价性**：分片计算的结果与完整矩阵计算完全一致，因为 `sum(h_i @ w2_i) = h @ W2`（矩阵分块乘法的性质）。
3. **All-reduce 是唯一的通信**：列并行的输出天然分片，行并行的输出需要求和，所以整个前向传播只有一次 all-reduce。
4. **GELU 在中间**：激活函数必须在列并行之后、行并行之前，因为每个分片独立计算自己的隐藏维度。